# Model Comparison — 270-Day Time-Based Holdout

This notebook evaluates all recommendation models using the same temporal evaluation window. Training data contains only purchases older than 270 days from the latest purchase date. The final 270 days are used only as the holdout period for evaluation. Seasonal matrices remain part of the recommendation rules where they were used originally; the hybrid model uses season, location, and age_group matrices.

In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from reactiva.config import DATASET_URI
from reactiva.features.build_features import build_features


In [4]:
# ============================================================
# LOAD DATA + SEASON / AGE-GROUP FEATURES + 270-DAY HOLDOUT
# ============================================================

df = pd.read_csv(DATASET_URI)
df['Purchase Date'] = pd.to_datetime(df['Purchase Date'])
df = build_features(df)

MAX_DATE = df['Purchase Date'].max()
CUTOFF = MAX_DATE - pd.Timedelta(days=270)

df_train = df[df['Purchase Date'] < CUTOFF].copy()
df_holdout = df[df['Purchase Date'] >= CUTOFF].copy()

train_seasons = df_train['season'].unique()
train_customers = df_train['Customer ID'].unique()
holdout_customers = df_holdout['Customer ID'].unique()
scoring_customers = np.intersect1d(train_customers, holdout_customers)

print(f'Latest purchase date: {MAX_DATE.date()}')
print(f'Cutoff date: {CUTOFF.date()}')
print(f'Train rows (older than cutoff): {len(df_train)}')
print(f'Holdout rows (last 270 days): {len(df_holdout)}')
print(f'Training seasons available: {sorted(train_seasons)}')
print(f'Scoring customers: {len(scoring_customers)}')


Latest purchase date: 2024-12-30
Cutoff date: 2024-04-04
Train rows (older than cutoff): 6281
Holdout rows (last 270 days): 3719
Training seasons available: ['monsoon', 'post-monsoon', 'summer', 'winter']
Scoring customers: 1877


In [5]:
# ============================================================
# SHARED METRIC HELPERS
# ============================================================

K_DEFAULT = 5


def precision_recall_at_k(recommended, actual, k):
    rec_k = list(recommended[:k])
    actual = set(actual)
    if not rec_k:
        return 0.0, 0.0, 0.0

    hits = len(set(rec_k) & actual)
    precision = hits / len(rec_k)
    recall = hits / len(actual) if actual else 0.0
    hit_rate = float(hits > 0)

    return precision, recall, hit_rate


def ndcg_at_k(recommended, actual, k):
    """Binary NDCG@K: correct/relevant items receive relevance=1."""
    rec_k = list(recommended[:k])
    actual = set(actual)
    if not rec_k or not actual:
        return 0.0

    dcg = 0.0
    for rank, item in enumerate(rec_k, start=1):
        if item in actual:
            dcg += 1.0 / np.log2(rank + 1)

    ideal_hits = min(len(actual), k)
    idcg = sum(
        1.0 / np.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )

    return dcg / idcg if idcg else 0.0


def average_precision_at_k(recommended, actual, k):
    """Average Precision@K for binary relevance."""
    rec_k = list(recommended[:k])
    actual = set(actual)
    if not rec_k or not actual:
        return 0.0

    hits = 0
    precision_sum = 0.0
    seen = set()

    for rank, item in enumerate(rec_k, start=1):
        if item in actual and item not in seen:
            seen.add(item)
            hits += 1
            precision_sum += hits / rank

    return precision_sum / min(len(actual), k) if min(len(actual), k) else 0.0


def long_tail_sets(df_train, cumulative_cutoff=0.8):
    """Define head/tail from training popularity only, avoiding holdout leakage."""
    counts = df_train['Item Purchased'].value_counts()
    cum_share = counts.cumsum() / counts.sum()
    head_items = set(cum_share[cum_share <= cumulative_cutoff].index)

    if not head_items and len(counts):
        head_items = {counts.index[0]}

    tail_items = set(counts.index) - head_items
    return head_items, tail_items


def long_tail_metrics(recommended, actual, tail_items, k):
    """Customer-level long-tail precision/recall/hit rate."""
    rec_k = list(recommended[:k])
    actual_tail = set(actual) & tail_items
    recommended_tail = set(rec_k) & tail_items
    hits = len(recommended_tail & actual_tail)

    # Recall/precision are defined only for customers with relevant tail items
    # when measuring tail performance. Share/coverage are handled separately.
    long_tail_precision = hits / len(recommended_tail) if recommended_tail else 0.0
    long_tail_recall = hits / len(actual_tail) if actual_tail else np.nan
    long_tail_hit_rate = float(hits > 0) if actual_tail else np.nan

    return long_tail_precision, long_tail_recall, long_tail_hit_rate


def get_result_columns(results_df):
    """Resolve recommendation/actual columns while preserving each model's existing names."""
    recommendation_candidates = [
        'recommended', 'Recommendations', 'recommendations'
    ]
    actual_candidates = [
        'actual', 'Actual', 'Actual Items'
    ]

    recommendation_col = next(
        (column for column in recommendation_candidates if column in results_df.columns),
        None
    )
    actual_col = next(
        (column for column in actual_candidates if column in results_df.columns),
        None
    )

    if recommendation_col is None or actual_col is None:
        raise ValueError('Could not identify recommendation/actual columns in results dataframe.')

    return recommendation_col, actual_col


def calculate_all_ranking_and_long_tail_metrics(
    results_df,
    df_train,
    k=5,
    cumulative_cutoff=0.8
):
    """Calculate ranking and long-tail metrics from stored recommendations/actuals."""
    if results_df.empty:
        return {
            'Precision': np.nan,
            'Recall': np.nan,
            'Hit Rate': np.nan,
            'NDCG': np.nan,
            'MAP': np.nan,
            'Long-tail Precision': np.nan,
            'Long-tail Recall': np.nan,
            'Long-tail Hit Rate': np.nan,
            'Long-tail Share': np.nan,
            'Long-tail Catalog Coverage': np.nan
        }

    recommendation_col, actual_col = get_result_columns(results_df)
    _, tail_items = long_tail_sets(
        df_train,
        cumulative_cutoff=cumulative_cutoff
    )

    precision_values = []
    recall_values = []
    hit_values = []
    ndcg_values = []
    map_values = []
    tail_precision_values = []
    tail_recall_values = []
    tail_hit_values = []
    all_recommended_items = []

    for _, row in results_df.iterrows():
        recommended = row[recommendation_col] or []
        actual = row[actual_col] or []

        precision, recall, hit_rate = precision_recall_at_k(
            recommended, actual, k
        )
        precision_values.append(precision)
        recall_values.append(recall)
        hit_values.append(hit_rate)
        ndcg_values.append(ndcg_at_k(recommended, actual, k))
        map_values.append(average_precision_at_k(recommended, actual, k))

        tail_precision, tail_recall, tail_hit = long_tail_metrics(
            recommended, actual, tail_items, k
        )
        tail_precision_values.append(tail_precision)
        tail_recall_values.append(tail_recall)
        tail_hit_values.append(tail_hit)

        all_recommended_items.extend(list(recommended[:k]))

    recommendation_denominator = len(all_recommended_items)
    tail_recommendation_count = sum(
        1 for item in all_recommended_items if item in tail_items
    )
    long_tail_share = (
        tail_recommendation_count / recommendation_denominator
        if recommendation_denominator else 0.0
    )

    distinct_tail_recommended = {
        item for item in all_recommended_items if item in tail_items
    }
    long_tail_catalog_coverage = (
        len(distinct_tail_recommended) / len(tail_items)
        if tail_items else 0.0
    )

    return {
        'Precision': np.mean(precision_values),
        'Recall': np.mean(recall_values),
        'Hit Rate': np.mean(hit_values),
        'NDCG': np.mean(ndcg_values),
        'MAP': np.mean(map_values),
        'Long-tail Precision': np.mean(tail_precision_values),
        'Long-tail Recall': np.nanmean(tail_recall_values),
        'Long-tail Hit Rate': np.nanmean(tail_hit_values),
        'Long-tail Share': long_tail_share,
        'Long-tail Catalog Coverage': long_tail_catalog_coverage
    }


def sparsity_of_matrix(user_item_matrix):
    total_cells = user_item_matrix.size
    if total_cells == 0:
        return np.nan

    nonzero_cells = (user_item_matrix.values > 0).sum()
    return 1 - (nonzero_cells / total_cells)


def get_popularity_scores(df_train):
    counts = df_train['Item Purchased'].value_counts()
    return counts / counts.max()


## 1. User-Based Collaborative Filtering

In [6]:
# ============================================================
# USER-BASED CF — SEASON MATRICES
# ============================================================

def backtest_recommender_pr(df_train, df_holdout, train_seasons, k=5):
    season_matrices = {}
    sparsities = []

    for s in train_seasons:
        df_tovector = df_train[df_train['season'] == s]
        user_item_matrix = pd.crosstab(
            df_tovector['Customer ID'],
            df_tovector['Item Purchased']
        )

        sparsity = sparsity_of_matrix(user_item_matrix)
        sparsities.append(sparsity)

        similarity = cosine_similarity(user_item_matrix)
        season_matrices[s] = pd.DataFrame(
            similarity,
            index=user_item_matrix.index,
            columns=user_item_matrix.index
        )

    results = []
    precision_scores = []
    recall_scores = []
    hit_scores = []

    for user in scoring_customers:
        user_dict = {}

        for s, similarity_df in season_matrices.items():
            if user not in similarity_df.columns:
                continue

            top5 = (
                similarity_df[user]
                .drop(user)
                .sort_values(ascending=False)
                .head(5)
            )

            for sim_user, _ in top5.items():
                if sim_user not in df_holdout['Customer ID'].values:
                    continue

                user_dict[sim_user] = df_holdout[
                    df_holdout['Customer ID'] == sim_user
                ]

        if len(user_dict) > 1:
            df_users = pd.concat(user_dict.values(), ignore_index=True)
            recommendation = (
                df_users['Item Purchased']
                .value_counts()
                .head(k)
                .index
                .tolist()
            )
        else:
            recommendation = []

        actual = set(
            df_holdout[
                df_holdout['Customer ID'] == user
            ]['Item Purchased']
        )

        precision, recall, hit_rate = precision_recall_at_k(
            recommendation, actual, k
        )

        precision_scores.append(precision)
        recall_scores.append(recall)
        hit_scores.append(hit_rate)

        results.append({
            'Customer ID': user,
            'recommended': recommendation,
            'actual': list(actual),
            'precision@k': precision,
            'recall@k': recall,
            'hit_rate@k': hit_rate
        })

    results_df = pd.DataFrame(results)

    print(f'Average sparsity across training seasons: {np.mean(sparsities):.4f}')
    print(f'Precision@{k}: {np.mean(precision_scores):.4f}')
    print(f'Recall@{k}: {np.mean(recall_scores):.4f}')
    print(f'HitRate@{k}: {np.mean(hit_scores):.4f}')

    return results_df, season_matrices, float(np.mean(sparsities))

results_user_based, season_matrices_user_based, sparsity_user_based = (
    backtest_recommender_pr(
        df_train=df_train,
        df_holdout=df_holdout,
        train_seasons=train_seasons,
        k=5
    )
)


Average sparsity across training seasons: 0.9486
Precision@5: 0.0938
Recall@5: 0.2725
HitRate@5: 0.3841


## 2. Frequency-Weighted User-Based Collaborative Filtering

In [7]:
# ============================================================
# FREQUENCY-WEIGHTED USER-BASED CF
# ============================================================

def build_customer_profile(df_train):
    customer_item_matrix = (
        df_train
        .groupby(['Customer ID', 'Item Purchased'])
        .size()
        .unstack(fill_value=0)
    )
    return customer_item_matrix

def build_customer_similarity(df_train):
    customer_item_matrix = build_customer_profile(df_train)
    similarity = cosine_similarity(customer_item_matrix)
    similarity_df = pd.DataFrame(
        similarity,
        index=customer_item_matrix.index,
        columns=customer_item_matrix.index
    )
    return similarity_df

def get_user_based_recommendations(customer_id, similarity_df, df_holdout, top_n=5, k=5):
    if customer_id not in similarity_df.index:
        return []

    neighbors = (
        similarity_df[customer_id]
        .drop(customer_id)
        .sort_values(ascending=False)
        .head(top_n)
    )

    if neighbors.empty:
        return []

    neighbor_ids = neighbors.index
    neighbor_purchases = df_holdout[
        df_holdout['Customer ID'].isin(neighbor_ids)
    ]

    if neighbor_purchases.empty:
        return []

    item_counts = neighbor_purchases['Item Purchased'].value_counts()
    return item_counts.head(k).index.tolist()

def backtest_frequency_user_based(df_train, df_holdout, top_n=5, k=5):
    train_customers = df_train['Customer ID'].unique()
    holdout_customers = df_holdout['Customer ID'].unique()
    scoring_customers = np.intersect1d(train_customers, holdout_customers)

    customer_item_matrix = build_customer_profile(df_train)
    similarity_df = build_customer_similarity(df_train)
    sparsity = sparsity_of_matrix(customer_item_matrix)

    results = []
    precision_scores = []
    recall_scores = []
    hit_scores = []

    for customer_id in scoring_customers:
        recommendation = get_user_based_recommendations(
            customer_id=customer_id,
            similarity_df=similarity_df,
            df_holdout=df_holdout,
            top_n=top_n,
            k=k
        )

        actual = set(
            df_holdout[
                df_holdout['Customer ID'] == customer_id
            ]['Item Purchased']
        )

        precision, recall, hit_rate = precision_recall_at_k(
            recommendation, actual, k
        )

        precision_scores.append(precision)
        recall_scores.append(recall)
        hit_scores.append(hit_rate)

        results.append({
            'Customer ID': customer_id,
            'recommended': recommendation,
            'actual': list(actual),
            'precision@k': precision,
            'recall@k': recall,
            'hit_rate@k': hit_rate
        })

    results_df = pd.DataFrame(results)

    print(f'Sparsity: {sparsity:.4f}')
    print(f'Precision@{k}: {np.mean(precision_scores):.4f}')
    print(f'Recall@{k}: {np.mean(recall_scores):.4f}')
    print(f'HitRate@{k}: {np.mean(hit_scores):.4f}')

    return results_df, similarity_df, customer_item_matrix, sparsity

results_frequency, similarity_df_frequency, customer_profile_frequency, sparsity_frequency = (
    backtest_frequency_user_based(
        df_train=df_train,
        df_holdout=df_holdout,
        top_n=5,
        k=5
    )
)


Sparsity: 0.9141
Precision@5: 0.0893
Recall@5: 0.2230
HitRate@5: 0.3239


## 3. Content-Based Recommendation

In [8]:
# ============================================================
# CONTENT-BASED MODEL — TRAIN ON df_train ONLY
# ============================================================

def build_content_profiles(df_train):
    customer_profiles = (
        df_train
        .groupby('Customer ID')['Item Purchased']
        .apply(lambda x: ' '.join(x.astype(str)))
    )

    vectorizer = TfidfVectorizer()
    customer_vectors = vectorizer.fit_transform(customer_profiles)

    similarity = cosine_similarity(customer_vectors)
    similarity_df = pd.DataFrame(
        similarity,
        index=customer_profiles.index,
        columns=customer_profiles.index
    )

    return similarity_df, customer_vectors, vectorizer

def backtest_content_based(df_train, df_holdout, top_n=5, k=5):
    train_customers = df_train['Customer ID'].unique()
    holdout_customers = df_holdout['Customer ID'].unique()
    scoring_customers = np.intersect1d(train_customers, holdout_customers)

    similarity_df, customer_vectors, vectorizer = build_content_profiles(df_train)

    customer_item_matrix = pd.crosstab(
        df_train['Customer ID'],
        df_train['Item Purchased']
    )
    sparsity = sparsity_of_matrix(customer_item_matrix)

    results = []
    precision_scores = []
    recall_scores = []
    hit_scores = []

    for customer_id in scoring_customers:
        if customer_id not in similarity_df.index:
            continue

        similar_customers = (
            similarity_df[customer_id]
            .drop(customer_id)
            .sort_values(ascending=False)
            .head(top_n)
        )

        if similar_customers.empty:
            recommendation = []
        else:
            neighbor_purchases = df_holdout[
                df_holdout['Customer ID'].isin(similar_customers.index)
            ]

            if neighbor_purchases.empty:
                recommendation = []
            else:
                recommendation = (
                    neighbor_purchases['Item Purchased']
                    .value_counts()
                    .head(k)
                    .index
                    .tolist()
                )

        actual = set(
            df_holdout[
                df_holdout['Customer ID'] == customer_id
            ]['Item Purchased']
        )

        precision, recall, hit_rate = precision_recall_at_k(
            recommendation, actual, k
        )

        precision_scores.append(precision)
        recall_scores.append(recall)
        hit_scores.append(hit_rate)

        results.append({
            'Customer ID': customer_id,
            'Similar Customers': similar_customers.index.tolist(),
            'Recommendations': recommendation,
            'Actual': list(actual),
            f'Precision@{k}': precision,
            f'Recall@{k}': recall,
            f'HitRate@{k}': hit_rate
        })

    results_df = pd.DataFrame(results)

    print(f'Sparsity: {sparsity:.4f}')
    print(f'Precision@{k}: {np.mean(precision_scores):.4f}')
    print(f'Recall@{k}: {np.mean(recall_scores):.4f}')
    print(f'HitRate@{k}: {np.mean(hit_scores):.4f}')

    return results_df, similarity_df, vectorizer, sparsity

results_content, similarity_df_content, vectorizer_content, sparsity_content = (
    backtest_content_based(
        df_train=df_train,
        df_holdout=df_holdout,
        top_n=5,
        k=5
    )
)


Sparsity: 0.9141
Precision@5: 0.0884
Recall@5: 0.2254
HitRate@5: 0.3277


## 4. Popularity Baseline

In [9]:
# ============================================================
# POPULARITY BASELINE
# ============================================================

def backtest_popularity(df_train, df_holdout, k=5):
    train_customers = df_train['Customer ID'].unique()
    holdout_customers = df_holdout['Customer ID'].unique()
    scoring_customers = np.intersect1d(train_customers, holdout_customers)

    user_item_matrix = pd.crosstab(
        df_train['Customer ID'],
        df_train['Item Purchased']
    )
    sparsity = sparsity_of_matrix(user_item_matrix)
    pop_scores = get_popularity_scores(df_train)

    results = []
    precision_scores = []
    recall_scores = []
    hit_scores = []

    for user in scoring_customers:
        recommendation = (
            pop_scores
            .sort_values(ascending=False)
            .head(k)
            .index
            .tolist()
        )

        actual = set(
            df_holdout[
                df_holdout['Customer ID'] == user
            ]['Item Purchased']
        )

        precision, recall, hit_rate = precision_recall_at_k(
            recommendation, actual, k
        )

        precision_scores.append(precision)
        recall_scores.append(recall)
        hit_scores.append(hit_rate)

        results.append({
            'Customer ID': user,
            'recommended': recommendation,
            'actual': list(actual),
            'precision@k': precision,
            'recall@k': recall,
            'hit_rate@k': hit_rate
        })

    results_df = pd.DataFrame(results)

    print(f'Sparsity: {sparsity:.4f}')
    print(f'Precision@{k}: {np.mean(precision_scores):.4f}')
    print(f'Recall@{k}: {np.mean(recall_scores):.4f}')
    print(f'HitRate@{k}: {np.mean(hit_scores):.4f}')

    return results_df, sparsity

results_popularity, sparsity_popularity = backtest_popularity(
    df_train=df_train,
    df_holdout=df_holdout,
    k=5
)


Sparsity: 0.9141
Precision@5: 0.1290
Recall@5: 0.4079
HitRate@5: 0.5440


## 5. Item-Item Collaborative Filtering

In [10]:
# ============================================================
# ITEM-ITEM CF — 270-DAY TEMPORAL HOLDOUT
# ============================================================

def build_item_item_similarity(df_train):
    customer_items = (
        df_train
        .groupby('Customer ID')['Item Purchased']
        .apply(set)
    )

    all_items = sorted(
        df_train['Item Purchased'].dropna().unique()
    )

    customer_item_matrix = pd.DataFrame(
        0,
        index=customer_items.index,
        columns=all_items,
        dtype=int
    )

    for customer_id, items in customer_items.items():
        for item in items:
            customer_item_matrix.loc[customer_id, item] = 1

    sparsity = sparsity_of_matrix(customer_item_matrix)

    item_customer_matrix = customer_item_matrix.T
    similarity = cosine_similarity(item_customer_matrix)
    similarity = pd.DataFrame(
        similarity,
        index=item_customer_matrix.index,
        columns=item_customer_matrix.index
    )

    return similarity, sparsity

def get_recommendations(trigger_item, similarity, top_n=5):
    if trigger_item not in similarity.index:
        return []

    scores = similarity.loc[trigger_item].copy()
    scores = scores.drop(labels=[trigger_item], errors='ignore')
    scores = scores[scores > 0]
    scores = scores.sort_values(ascending=False)

    return scores.head(top_n).index.tolist()

def precision_at_k(recommendations, actual_items, k):
    recommendations = recommendations[:k]
    if not recommendations:
        return 0.0

    hits = len(set(recommendations) & set(actual_items))
    return hits / len(recommendations)

def recall_at_k(recommendations, actual_items, k):
    recommendations = recommendations[:k]
    actual_items = set(actual_items)

    if not actual_items:
        return 0.0

    hits = len(set(recommendations) & actual_items)
    return hits / len(actual_items)

def hit_rate_at_k(recommendations, actual_item, k):
    return int(actual_item in recommendations[:k])

def evaluate_item_item_cf(df_train, df_holdout, k=5):
    results = []
    precisions = []
    recalls = []
    hit_rates = []

    similarity, sparsity = build_item_item_similarity(df_train)

    for customer_id, customer_holdout in df_holdout.groupby('Customer ID'):
        customer_train = df_train[
            df_train['Customer ID'] == customer_id
        ]

        if customer_train.empty:
            continue

        trigger_item = (
            customer_train
            .sort_values('Purchase Date')['Item Purchased']
            .iloc[-1]
        )

        actual_items = set(customer_holdout['Item Purchased'].unique())
        if not actual_items:
            continue

        recommendations = get_recommendations(
            trigger_item,
            similarity,
            top_n=k
        )

        precision = precision_at_k(
            recommendations, actual_items, k
        )
        recall = recall_at_k(
            recommendations, actual_items, k
        )

        actual_item = (
            customer_holdout
            .sort_values('Purchase Date')['Item Purchased']
            .iloc[0]
        )

        hit_rate = hit_rate_at_k(
            recommendations, actual_item, k
        )

        precisions.append(precision)
        recalls.append(recall)
        hit_rates.append(hit_rate)

        results.append({
            'Customer ID': customer_id,
            'Trigger': trigger_item,
            'Actual Items': list(actual_items),
            'First Actual Item': actual_item,
            'Recommendations': recommendations,
            f'Precision@{k}': precision,
            f'Recall@{k}': recall,
            f'HitRate@{k}': hit_rate
        })

    print(f'Sparsity: {sparsity:.4f}')
    print(f'Precision@{k}: {np.mean(precisions) if precisions else 0.0:.4f}')
    print(f'Recall@{k}: {np.mean(recalls) if recalls else 0.0:.4f}')
    print(f'HitRate@{k}: {np.mean(hit_rates) if hit_rates else 0.0:.4f}')

    return pd.DataFrame(results), sparsity

results_item_item, sparsity_item_item = evaluate_item_item_cf(
    df_train=df_train,
    df_holdout=df_holdout,
    k=5
)


Sparsity: 0.9141
Precision@5: 0.1139
Recall@5: 0.3565
HitRate@5: 0.3484


## 6. Gradient Boosting Classification

In [11]:
# ============================================================
# GRADIENT BOOSTING CLASSIFICATION — 270-DAY HOLDOUT
# ============================================================

def build_customer_features(df_train):
    cat_counts = df_train.pivot_table(
        index='Customer ID',
        columns='Category',
        values='Item Purchased',
        aggfunc='count',
        fill_value=0
    )
    cat_counts.columns = [
        f'cat_count_{c}' for c in cat_counts.columns
    ]

    agg = df_train.groupby('Customer ID').agg(
        total_purchases=('Item Purchased', 'count'),
        last_purchase=('Purchase Date', 'max')
    )

    reference_date = df_train['Purchase Date'].max()
    agg['days_since_last_purchase'] = (
        reference_date - agg['last_purchase']
    ).dt.days
    agg = agg.drop(columns='last_purchase')

    return cat_counts.join(agg, how='inner')

def backtest_gradient_boosting(df_train, df_holdout, k=5, test_size=0.3, random_state=42):
    train_customers = df_train['Customer ID'].unique()
    holdout_customers = df_holdout['Customer ID'].unique()
    scoring_customers = np.intersect1d(train_customers, holdout_customers)

    features = build_customer_features(df_train)
    features = features.loc[features.index.isin(scoring_customers)]

    labels = (
        df_holdout[
            df_holdout['Customer ID'].isin(features.index)
        ]
        .groupby('Customer ID')['Category']
        .agg(lambda x: x.mode().iloc[0])
    )

    X = features
    y = labels.loc[X.index]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    clf = GradientBoostingClassifier(
        random_state=random_state
    )
    clf.fit(X_train, y_train)

    pred_category = pd.Series(
        clf.predict(X_test),
        index=X_test.index
    )

    item_pop_by_cat = (
        df_train
        .groupby(['Category', 'Item Purchased'])
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
    )

    precision_scores = []
    recall_scores = []
    hit_scores = []
    recommendation_results = []

    for user in X_test.index:
        cat = pred_category[user]

        top_items = (
            item_pop_by_cat[
                item_pop_by_cat['Category'] == cat
            ]['Item Purchased']
            .head(k)
            .tolist()
        )

        actual = set(
            df_holdout[
                df_holdout['Customer ID'] == user
            ]['Item Purchased']
        )

        hits = len(set(top_items) & actual)
        precision = hits / len(top_items) if top_items else 0.0
        recall = hits / len(actual) if actual else 0.0
        hit_rate = int(hits > 0)

        precision_scores.append(precision)
        recall_scores.append(recall)
        hit_scores.append(hit_rate)

        recommendation_results.append({
            'Customer ID': user,
            'Predicted Category': cat,
            'Recommendations': top_items,
            'Actual Items': list(actual),
            'Hits': hits,
            f'Precision@{k}': precision,
            f'Recall@{k}': recall,
            f'HitRate@{k}': hit_rate
        })

    precision_at_k_mean = np.mean(precision_scores)
    recall_at_k_mean = np.mean(recall_scores)
    hit_rate_mean = np.mean(hit_scores)

    results = pd.DataFrame(recommendation_results)

    print(f'Gradient Boosting results @ {k}')
    print(f'Precision@{k}: {precision_at_k_mean:.4f}')
    print(f'Recall@{k}: {recall_at_k_mean:.4f}')
    print(f'HitRate@{k}: {hit_rate_mean:.4f}')

    return clf, hit_rate_mean, precision_at_k_mean, recall_at_k_mean, results

clf, hit_rate_classification, precision_classification, recall_classification, results_classification = (
    backtest_gradient_boosting(
        df_train=df_train,
        df_holdout=df_holdout,
        k=5,
        test_size=0.3,
        random_state=42
    )
)


Gradient Boosting results @ 5
Precision@5: 0.1078
Recall@5: 0.3418
HitRate@5: 0.4681


## 7. Hybrid User-Based CF + Popularity Fallback

In [ ]:
# ============================================================
# HYBRID — SEASON + LOCATION + AGE_GROUP MATRICES
# ============================================================

THRESHOLD = 0.0

def classify_head_tail(df_train, cumulative_cutoff=0.8):
    counts = df_train['Item Purchased'].value_counts()
    cum_share = counts.cumsum() / counts.sum()
    head_items = set(
        cum_share[cum_share <= cumulative_cutoff].index
    )

    if not head_items:
        head_items = {counts.index[0]}

    tail_items = set(counts.index) - head_items
    return head_items, tail_items

def get_cf_scores_grouped(user, group_matrices, df_holdout, top_n=5):
    user_dict = {}

    for key, similarity_df in group_matrices.items():
        if user not in similarity_df.columns:
            continue

        top_neighbors = (
            similarity_df[user]
            .drop(user)
            .sort_values(ascending=False)
            .head(top_n)
        )

        for sim_user, _ in top_neighbors.items():
            if sim_user not in df_holdout['Customer ID'].values:
                continue

            user_dict[sim_user] = df_holdout[
                df_holdout['Customer ID'] == sim_user
            ]

    if not user_dict:
        return pd.Series(dtype=float)

    df_users = pd.concat(
        user_dict.values(),
        ignore_index=True
    )
    counts = df_users['Item Purchased'].value_counts()

    return counts / counts.max() if len(counts) else pd.Series(dtype=float)

def backtest_hybrid_avg_threshold(df_train, df_holdout, k=5, top_n=5):
    train_customers = df_train['Customer ID'].unique()
    holdout_customers = df_holdout['Customer ID'].unique()
    scoring_customers = np.intersect1d(train_customers, holdout_customers)

    group_matrices = {}
    sparsity_by_group = []

    group_keys = df_train[
        ['season', 'Location', 'age_group']
    ].drop_duplicates()

    for _, row in group_keys.iterrows():
        s = row['season']
        loc = row['Location']
        ag = row['age_group']

        df_group = df_train[
            (df_train['season'] == s)
            & (df_train['Location'] == loc)
            & (df_train['age_group'] == ag)
        ]

        if df_group['Customer ID'].nunique() < 2:
            continue

        user_item_matrix = pd.crosstab(
            df_group['Customer ID'],
            df_group['Item Purchased']
        )
        sparsity_by_group.append(
            sparsity_of_matrix(user_item_matrix)
        )

        similarity = cosine_similarity(user_item_matrix)
        group_matrices[(s, loc, ag)] = pd.DataFrame(
            similarity,
            index=user_item_matrix.index,
            columns=user_item_matrix.index
        )

    pop_scores = get_popularity_scores(df_train)
    head_items, tail_items = classify_head_tail(df_train)

    results = []
    precision_scores = []
    recall_scores = []
    hit_scores = []
    all_recommended_items = []
    cf_used = 0
    pop_used = 0

    for user in scoring_customers:
        all_top_neighbor_sims = []

        for key, similarity_df in group_matrices.items():
            if user not in similarity_df.columns:
                continue

            neighbor_sims = (
                similarity_df[user]
                .drop(user)
                .sort_values(ascending=False)
                .head(top_n)
            )
            all_top_neighbor_sims.extend(
                neighbor_sims.tolist()
            )

        avg_sim = (
            np.mean(all_top_neighbor_sims)
            if all_top_neighbor_sims
            else -1
        )

        if avg_sim >= THRESHOLD:
            cf_scores = get_cf_scores_grouped(
                user,
                group_matrices,
                df_holdout,
                top_n=top_n
            )

            if len(cf_scores):
                recommendation = (
                    cf_scores
                    .sort_values(ascending=False)
                    .head(k)
                    .index
                    .tolist()
                )
                cf_used += 1
            else:
                recommendation = (
                    pop_scores
                    .head(k)
                    .index
                    .tolist()
                )
                pop_used += 1
        else:
            recommendation = (
                pop_scores
                .head(k)
                .index
                .tolist()
            )
            pop_used += 1

        all_recommended_items.extend(recommendation[:k])

        actual = set(
            df_holdout[
                df_holdout['Customer ID'] == user
            ]['Item Purchased']
        )

        precision, recall, hit_rate = precision_recall_at_k(
            recommendation, actual, k
        )

        precision_scores.append(precision)
        recall_scores.append(recall)
        hit_scores.append(hit_rate)

        results.append({
            'Customer ID': user,
            'avg_neighbor_sim': avg_sim,
            'recommendations': recommendation,
            'actual': list(actual),
            'precision@k': precision,
            'recall@k': recall,
            'hit_rate@k': hit_rate
        })

    results_df = pd.DataFrame(results)

    hybrid_group_sparsity = (
        np.mean(sparsity_by_group)
        if sparsity_by_group
        else np.nan
    )

    distinct_recommended = set(all_recommended_items)
    tail_impressions = sum(
        1 for item in all_recommended_items
        if item in tail_items
    )
    long_tail_share = (
        tail_impressions / len(all_recommended_items)
        if all_recommended_items
        else 0.0
    )
    catalog_coverage = (
        len(distinct_recommended)
        / df_train['Item Purchased'].nunique()
    )

    print(f'Hybrid threshold: {THRESHOLD:.2f}')
    print(f'Precision@{k}: {np.mean(precision_scores):.4f}')
    print(f'Recall@{k}: {np.mean(recall_scores):.4f}')
    print(f'HitRate@{k}: {np.mean(hit_scores):.4f}')
    print(f'Customers routed to CF: {cf_used}')
    print(f'Customers routed to popularity fallback: {pop_used}')
    print(f'Groups built: {len(group_matrices)}')
    print(f'Hybrid sparsity: {hybrid_group_sparsity:.4f}')
    print(f'Long-tail share: {long_tail_share:.4f}')
    print(f'Catalog coverage: {catalog_coverage:.4f}')

    return (
        results_df,
        group_matrices,
        hybrid_group_sparsity,
        long_tail_share,
        catalog_coverage
    )

results_hybrid, group_matrices_hybrid, sparsity_hybrid, long_tail_share_hybrid, catalog_coverage_hybrid = (
    backtest_hybrid_avg_threshold(
        df_train=df_train,
        df_holdout=df_holdout,
        k=5,
        top_n=5
    )
)


Hybrid threshold: 0.30
Precision@5: 0.0952
Recall@5: 0.2896
HitRate@5: 0.4033
Customers routed to CF: 1362
Customers routed to popularity fallback: 515
Groups built: 160
Hybrid sparsity: 0.8932
Long-tail share: 0.1425
Catalog coverage: 1.0000


## Final Model Comparison

In [13]:
# ============================================================
# FINAL EVALUATION TABLE — SAME 270-DAY HOLDOUT FOR ALL MODELS
# ============================================================

# Metric glossary:
# Precision@K            = fraction of the K recommendations that were purchased.
# Recall@K               = fraction of the customer's purchased items captured by the K recommendations.
# Hit Rate@K             = whether at least one correct item appears in the K recommendations.
# NDCG@K                 = ranking quality; correct items receive more credit when ranked higher.
# MAP@K                  = ranking quality based on precision at each relevant position.
# Long-tail Precision@K  = precision considering only long-tail recommendations.
# Long-tail Recall@K     = fraction of the customer's actually purchased long-tail items captured.
# Long-tail Hit Rate@K   = whether at least one actually purchased long-tail item was recommended.
# Long-tail Share         = fraction of all recommendation impressions that are long-tail items.
# Long-tail Catalog Coverage = fraction of the training-defined long-tail catalog ever recommended.
# Average Score           = mean of the eight performance metrics above; long-tail share/coverage are diagnostics.
# Sparsity                = fraction of zero cells in the relevant customer-item matrix.

K = 5
TAIL_CUMULATIVE_CUTOFF = 0.8

metric_results = {
    'User-based': calculate_all_ranking_and_long_tail_metrics(
        results_user_based, df_train, k=K,
        cumulative_cutoff=TAIL_CUMULATIVE_CUTOFF
    ),
    'Frequency-weighted User-based': calculate_all_ranking_and_long_tail_metrics(
        results_frequency, df_train, k=K,
        cumulative_cutoff=TAIL_CUMULATIVE_CUTOFF
    ),
    'Content-based': calculate_all_ranking_and_long_tail_metrics(
        results_content, df_train, k=K,
        cumulative_cutoff=TAIL_CUMULATIVE_CUTOFF
    ),
    'Popularity': calculate_all_ranking_and_long_tail_metrics(
        results_popularity, df_train, k=K,
        cumulative_cutoff=TAIL_CUMULATIVE_CUTOFF
    ),
    'Item-based CF': calculate_all_ranking_and_long_tail_metrics(
        results_item_item, df_train, k=K,
        cumulative_cutoff=TAIL_CUMULATIVE_CUTOFF
    ),
    'Classification': calculate_all_ranking_and_long_tail_metrics(
        results_classification, df_train, k=K,
        cumulative_cutoff=TAIL_CUMULATIVE_CUTOFF
    ),
    'Hybrid': calculate_all_ranking_and_long_tail_metrics(
        results_hybrid, df_train, k=K,
        cumulative_cutoff=TAIL_CUMULATIVE_CUTOFF
    )
}

sparsity_results = {
    'User-based': sparsity_user_based,
    'Frequency-weighted User-based': sparsity_frequency,
    'Content-based': sparsity_content,
    'Popularity': sparsity_popularity,
    'Item-based CF': sparsity_item_item,
    'Classification': sparsity_of_matrix(
        pd.crosstab(df_train['Customer ID'], df_train['Item Purchased'])
    ),
    'Hybrid': sparsity_hybrid
}

performance_columns = [
    'Precision',
    'Recall',
    'Hit Rate',
    'Long-tail Precision',
    'Long-tail Recall',
    'Long-tail Hit Rate',
    'NDCG',
    'MAP'
]

rows = []
for model_name, metrics in metric_results.items():
    row = {
        'Model': model_name,
        **metrics,
        'Average Score': np.nanmean([
            metrics[column] for column in performance_columns
        ]),
        'Sparsity': sparsity_results[model_name]
    }
    rows.append(row)

evaluation_matrix = pd.DataFrame(rows)[
    [
        'Model',
        'Precision',
        'Recall',
        'Hit Rate',
        'Long-tail Precision',
        'Long-tail Recall',
        'Long-tail Hit Rate',
        'Long-tail Share',
        'Long-tail Catalog Coverage',
        'NDCG',
        'MAP',
        'Average Score',
        'Sparsity'
    ]
]

print('=== MODEL EVALUATION — 270-DAY TIME-BASED HOLDOUT ===')
print('Long-tail is defined from df_train using an 80% cumulative purchase-share cutoff.')
print('Average Score excludes Long-tail Share, Long-tail Catalog Coverage, and Sparsity because they are diagnostic/distribution measures.')
display(evaluation_matrix.round(4))


=== MODEL EVALUATION — 270-DAY TIME-BASED HOLDOUT ===
Long-tail is defined from df_train using an 80% cumulative purchase-share cutoff.
Average Score excludes Long-tail Share, Long-tail Catalog Coverage, and Sparsity because they are diagnostic/distribution measures.


,Model,Precision,Recall,Hit Rate,Long-tail Precision,Long-tail Recall,Long-tail Hit Rate,Long-tail Share,Long-tail Catalog Coverage,NDCG,MAP,Average Score,Sparsity
0,User-based,0.0938,0.2725,0.3841,0.0266,0.1173,0.1248,0.2012,1.0000,0.1839,0.1341,0.1671,0.9486
1,Frequency-weighted User-based,0.0893,0.2230,0.3239,0.0222,0.0909,0.0945,0.1912,1.0000,0.1565,0.1147,0.1394,0.9141
2,Content-based,0.0884,0.2254,0.3277,0.0282,0.1160,0.1248,0.2049,1.0000,0.1611,0.1197,0.1489,0.9141
3,Popularity,0.1290,0.4079,0.5440,0.0000,0.0000,0.0000,0.0000,0.0000,0.2729,0.2008,0.1943,0.9141
4,Item-based CF,0.1139,0.3565,0.4928,0.0069,0.0196,0.0232,0.0312,0.3333,0.2452,0.1810,0.1799,0.9141
5,Classification,0.1078,0.3418,0.4681,0.0013,0.0175,0.0175,0.0227,0.4444,0.2371,0.1770,0.1710,0.9141
6,Hybrid,0.0952,0.2896,0.4033,0.0132,0.0554,0.0624,0.1425,1.0000,0.1996,0.1480,0.1583,0.8932


### Evaluation window and metric notes

All models use the same temporal split: purchases before `CUTOFF` are training data, and purchases from `CUTOFF` through the latest purchase date are the 270-day holdout. Seasonal information is retained inside the recommendation logic where applicable; it is not used to define the holdout period.

**Long-tail:** tail items are defined from `df_train` only, using the cumulative 80% purchase-share cutoff. Long-tail Precision/Recall/Hit Rate evaluate customer-level performance on tail purchases; Long-tail Share measures how many recommendation impressions are tail items; Long-tail Catalog Coverage measures how much of the tail catalog is recommended.

**NDCG@K:** rewards relevant items more when they appear higher in the ranked list. **MAP@K:** averages precision at the ranks where relevant items occur.

**Average Score:** arithmetic mean of Precision, Recall, Hit Rate, Long-tail Precision, Long-tail Recall, Long-tail Hit Rate, NDCG, and MAP. Long-tail Share, Long-tail Catalog Coverage, and Sparsity remain separate diagnostics.


## Model Selection: User-Based Collaborative Filtering

Although the **Popularity** model achieved the highest scores on the traditional recommendation metrics—**Precision, Recall, Hit Rate, NDCG, and MAP**—the **User-Based Collaborative Filtering** model was selected because it provides a better balance between predictive performance, personalization, and recommendation diversity.

### Performance

The **User-Based Collaborative Filtering** model achieved:

- **Precision:** 0.0938
- **Recall:** 0.2725
- **Hit Rate:** 0.3841
- **NDCG:** 0.1839
- **MAP:** 0.1341

While these results are lower than those of the Popularity model, the User-Based approach performs substantially better when recommending **long-tail items**:

- **Long-tail Precision:** 0.0266
- **Long-tail Recall:** 0.1173
- **Long-tail Hit Rate:** 0.1248
- **Long-tail Share:** 0.2012
- **Long-tail Catalog Coverage:** 1.0000

### Reason for Selection

The results show that the User-Based model is able to recommend not only popular products but also less frequently purchased items across the entire long-tail catalog.

In contrast, although the **Popularity** model achieved the best overall predictive performance, it obtained **0 in all long-tail metrics**. This indicates that its recommendations are heavily concentrated on the most popular products and provide limited personalization or product diversity.

Therefore, **User-Based Collaborative Filtering** was selected as the final model. The decision prioritizes a balance between recommendation accuracy, personalization, diversity, and the ability to expose customers to less popular products, rather than optimizing only for overall Precision and Recall.